In [ ]:
import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
from PermCell_Smooth import *
#from SHAPset import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
Run="Corrs"
import xgboost as xgb
import umap
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import pandas as pd
from POgSET import *
import icecream as ic

In [ ]:
%matplotlib inline

In [ ]:
import glob

In [ ]:
metals_names_map = {' In115Di':'H3', ' Ce140Di':'Cytokeratin5', ' Nd142Di':'H3K27me2', ' Nd143Di':'p53', ' Nd144Di':'EZH2', ' Nd145Di':'H3K4me3',
                   ' Sm149Di':'H3K36me2', ' Nd150Di':'H3K4me1', ' Eu151Di':'H3K9me2', ' Sm152Di':'H4K16ac', ' Eu153Di':'H2AK119Ub', ' Gd155Di':'H3.3',
                   ' Gd156Di':'H3K64ac', ' Gd158Di':'ZEB1', ' Tb159Di':'H4', ' Gd160Di':'H3K27ac', ' Dy161Di':'H4K20me3', ' Ho165Di':'H3K36me3',
                   ' Er168Di':'H3K27me3', ' Tm169Di':'H3K9ac', ' Er170Di':'H3K9me3', ' Lu175Di':'H3S28p', ' Pr141Di':'human-EpCAM',
                    ' Sm147Di':'yH2A.X', ' Sm154Di':'Vimentin', ' Dy163Di':'ER', ' Dy164Di':'CD49f', ' Er166Di':'CD24', ' Er167Di':'GATA3',
                   ' Yb171Di':'CD44', ' Yb172Di':'Ki-67', ' Yb174Di':'K8_18'}

In [ ]:
R1=dict([(f[0][1:],f[1]) for f in metals_names_map.items()])

In [ ]:
R1

In [ ]:
dir="/Users/ronguy/Dropbox/CyTOF_Breast/202509_CyTOF-UCell/PermCell_on_existing_UMAPs/"

In [ ]:
F=glob.glob(dir+"*")

In [ ]:
F

In [ ]:
dir="PDXData/"
F=glob.glob(dir+"*")

In [ ]:
F.sort()
F

In [ ]:
F=[f for f in F if "_7_" in f]
F

In [ ]:
X_2d=pd.read_parquet(F[1]).values
DF=pd.read_parquet(F[0])

In [ ]:
pd.read_parquet(F[1])

In [ ]:
DF.columns

In [ ]:
DF

In [ ]:
#dir="/Users/ronguy/Dropbox/WIS-CIMA colab - Analysis/#3 CyTOF  - KPC sample, after CD45 depletion/"

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
from lmfit import minimize, Parameters

In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
%matplotlib inline

In [ ]:
NamesAll=list(DF.columns)

In [ ]:

NamesAll=['NCad',
 'ECad',
 'panKeratin',
 'K5',
 'EpCam',
 'H3K27me2',
 'EZH2',
 'gH2AX',
 'aSMA',
 'H3K36me2',
 'H3K4me1',
 'H3K9me2',
 'H4K16ac',
 'H2Aub',
 'Vimentin',
 'H3K64ac',
 'BMI-1',
 'H3K27ac',
 'H4K20me3',
 'ER',
 'CD49f',
 'CD24',
 'GATA3',
 'H3K9ac',
 'H3K9me3',
 'CD44',
 'Ki67',
 'K8-18',
 'H3K36me3',
 'H3K4me3',
 'H3K27me3',
 'H3S28p',
 'H4',
 'H3',
 'H3.3']

In [ ]:
NamesAll=['ECad', 'panKeratin', 'K5', 'EpCam', 'H3K27me2', 'gH2AX', 'aSMA',
       'H3K36me2', 'H3K4me1', 'H3K9me2', 'H4K16ac', 'H2Aub', 'Vimentin',
       'H3K64ac', 'BMI-1', 'H3K27ac', 'H4K20me3', 'ER', 'CD49f', 'CD24',
       'GATA3', 'H3K9ac', 'H3K9me3', 'CD44', 'Ki67', 'K8-18', 'H3K36me3',
       'H3K4me3', 'H3K27me3', 'MBD', 'H3S28p', 'samp', 'ind', 'H4', 'H3',
       'H3.3']

In [ ]:
DF=(DF[NamesAll]-DF[NamesAll].mean())/DF[NamesAll].std()

In [ ]:
DF.std()

In [ ]:
AD=ad.AnnData(DF,obsm={"X_umap":X_2d})

In [ ]:
NamesAll.sort()

In [ ]:
MRK=NamesAll.copy()
MRK.remove('H3')
MRK.remove('H3.3')
MRK.remove('H4')

In [ ]:
sc.pl.umap(AD,color=MRK,cmap='seismic',vmin='p1', vmax='p99',ncols=5)

In [ ]:
marker_sets = {
    # Luminal/epithelial program: epithelial & ER axis up; mesenchymal/basal down
    "Epithelial_Luminal": {
        "up":   {"EpCam", "ER", "ECad","GATA3",  "K8-18"}, #"panKeratin",
        "down": {"K5", "Vimentin", "aSMA"}
     },

    # # Basal-like program: basal keratin/mesenchymal up; luminal/epithelial down
    # "Basal_like": {
    #     "up":   {"KRT5",  "Vimentin"},
    #     "down": {"EpCAM", "E-cadherin", "ER", "GATA3", "Pan-KRT", "KRT8-18"}
    # },

    # “Basal_Noa” (histone-flavor): active H3K4 marks up; repressive H3K9me2 down
    "Basal_Noa": {
        "up":   {"H3K4me1", "H3K4me3","H3K9me2"},
        "down": {"H4K20me3","H3K36me3"}
    },

    # # EMT programs: Vimentin/aSMA/CD44 up; E-cadherin down
    # "EMT": {
    #     "up":   {"Vimentin", "aSMA", "CD44"},
    #     "down": {"E-cadherin"}
    # },
    

    # # Proliferation / cell-cycle & immediate-early signaling up
    # "Proliferation": {
    #     "up":   {"KI67", "H3S28p", "H3K9ac", "H3K64ac"},
    #     "down": set()  
    # },
}


marker_sets_extra = {
    # ---- PAM50-ish refinements ----
    "LuminalA_like": {
        "up":   {"ER", "GATA3", "KRT8-18", "E-cadherin", "EpCAM", "Pan-KRT"},
        "down": {"KRT5", "Vimentin", "aSMA", "CD44", "KI67"}  # lower proliferation bias
    },
    "LuminalB_like": {
        "up":   {"ER", "GATA3", "KRT8-18", "E-cadherin", "EpCAM", "Pan-KRT", "KI67"},
        "down": {"KRT5", "Vimentin", "aSMA"}
    },

    # ---- Hierarchy / progenitors ----
    "Luminal_Progenitor": {  # LP: EpCAM+, CD49f+, luminal keratins
        "up":   {"EpCAM", "CD49f", "KRT8-18", "Pan-KRT"},
        "down": {"KRT5", "Vimentin", "aSMA"}
    },
    "Basal_Progenitor": {    # basal/myo-biased progenitor
        "up":   {"CD49f", "KRT5", "CD44"},
        "down": {"EpCAM", "KRT8-18", "E-cadherin"}
    },
    "Myoepithelial_like": {
        "up":   {"aSMA", "KRT5", "CD49f", "Vimentin"},
        "down": {"KRT8-18", "EpCAM", "E-cadherin", "ER"}
    },

    # ---- Stemness / plasticity ----
    "CSC_CD44hi_CD24lo": {
        "up":   {"CD44", "CD49f", "BMI1", "EZH2"},
        "down": {"CD24", "E-cadherin", "ER", "GATA3"}
    },
    "Partial_EMT": {  # pEMT: epithelial retained with mesenchymal gain
        "up":   {"Vimentin", "CD44"},
        "down": set()  # keep neutral on E-cadherin here; use EMT set for strict E-cad↓
    },

    # ---- Proliferation & stress ----
    "High_Proliferation": {
        "up":   {"KI67", "H3S28p"},
        "down": set()
    },
    "DNA_Damage_Stress": {
        "up":   {"pH2A.X", "H3S28p"},
        "down": set()
    },

    # ---- Chromatin programs ----
    "Active_Chromatin": {
        "up":   {"H3K27ac", "H3K4me3", "H3K9ac", "H3K64ac", "H4K16ac", "H3K4me1"},
        "down": {"H3K27me3", "H3K9me3", "H4K20me3", "H3K27me2", "H3K9me2"}
    },
    "PRC_Repression": {  # PRC1/2 axis
        "up":   {"H3K27me3", "H2AK119ub", "EZH2", "BMI1"},
        "down": {"H3K27ac", "H3K4me3", "H3K9ac", "H4K16ac"}
    },
    "Active_Enhancer": {
        "up":   {"H3K4me1", "H3K27ac"},
        "down": {"H3K27me3"}
    },
    "Poised_Enhancer": {
        "up":   {"H3K4me1", "H3K27me3"},
        "down": {"H3K27ac"}
    },
    "Transcription_Elongation": {
        "up":   {"H3K36me3", "H3K36me2"},
        "down": set()
    },

    # ---- Epithelial integrity / adhesion ----
    "Epithelial_Adhesion": {
        "up":   {"E-cadherin", "EpCAM", "Pan-KRT", "KRT8-18"},
        "down": {"Vimentin", "aSMA", "KRT5"}
    },


}


In [ ]:
MRK=NamesAll.copy()
MRK.remove('H3')
MRK.remove('H3.3')
MRK.remove('H4')

In [ ]:
from icecream import ic
PLT={'Cycling':'#3f78c1','Basal-like':'#fb9a99','Basal':'#fb9a99','Luminal':'#33a02c','M':'gray','DNA Damage':'gray','G0':'black'}

In [ ]:
Z, P, Zabs, Pabs, Zdir = sipsic_like_scores_v3(
    AD, marker_sets,
    n_perm=2048,                   # or even 16 for smoke test
    prefer_permutation=True,     # <- important
    perm_batch=512,              # keeps RAM flat
    normalize_set_weights="l2",
    use_sparse_W=False,
    progress=True               # progress bars can add overhead in some envs
)

In [ ]:
Z

In [ ]:
DB=f'DF{F[0].split("/")[-1].split("_")[1]}'
DB='PDXs'

In [ ]:

sdf=pd.DataFrame(gaussian_smooth_all_torch(Z.values,AD.obsm['X_umap'],-1),columns=Z.columns)

ADS=ad.AnnData(obs=sdf)
ADS.obsm['X_umap']=AD.obsm['X_umap']

sc.pl.umap(AD,color=MRK,cmap='seismic',show=False,vmin='p1',vmax='p99')
plt.savefig(f"Plots/{DB}_UMAP.pdf",dpi=200,bbox_inches='tight')
sc.pl.umap(ADS,color=list(ADS.obs.columns),cmap='seismic',show=False,vcenter=0,palette=PLT)
plt.savefig(f"Plots/{DB}_UMAP_PermCell_with_H4K20me3.pdf",dpi=200,bbox_inches='tight')

ADUS=ad.AnnData(obs=Z)
ADUS.obsm['X_umap']=AD.obsm['X_umap']

sc.pl.umap(ADUS,color=list(ADUS.obs.columns),cmap='seismic',show=False,vcenter=0,palette=PLT)
plt.savefig(f"Plots/{DB}_UMAP_PermCell_Unsmoothed_with_H4K20me3.pdf",dpi=200,bbox_inches='tight')


plt.show()

In [ ]:
X_2d=ADUS.obsm['X_umap']
pd.DataFrame(X_2d,columns=['umap1','umap2']).to_parquet("UMAP_DF4.1.parquet")
#ADUS.obs

In [ ]:
ADUS.obs.to_parquet("DF4.1_Sigs.parquet")